# Sentiment Analysis for Customer Feedback — Airline Industry

**Case Study Part 1 · Research notebook**


## Introduction

Customer feedback can help airlines identify service concerns and prioritize responses. This research uses the existing Twitter US Airline Sentiment dataset to classify tweets as negative, neutral or positive and compare approaches for short customer-feedback text. Its objectives are to evaluate classification performance and examine how sentiment predictions could support complaint triage and service monitoring.

The workflow includes text cleaning, tokenization, lemmatization and stop-word removal while retaining negation cues. TF-IDF and Word2Vec features support classical models, which are compared with a BiLSTM. Accuracy, precision, recall and F1-score are assessed on a shared test set. Transformer code is included as an optional extension; completed Transformer results are unavailable.


## 0. Setup


In [ ]:
# --- Run-time switches -------------------------------------------------------
RUN_GRID_SEARCH = False    # True → re-run 5-fold grid search for every classical model (~20 min)
TRAIN_BILSTM = False       # True → train the BiLSTM here instead of loading saved predictions
TRAIN_TRANSFORMER = False  # True → fine-tune DistilRoBERTa here (~1 h on CPU, needs ~3 GB free RAM)

In [ ]:
import json
import sys
import time
import warnings
from pathlib import Path

# Make the project root importable whether Jupyter was started in the root or in notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from src import config, eda, evaluation as ev, insights
from src.classical_models import decision_scores, get_model_specs, tune_and_fit
from src.data_loading import clean_dataset, load_raw, split_data, summary_statistics
from src.features import Word2VecVectorizer, project_feature_space, word_tfidf
from src.plotting import MODEL_ORDER
from src.preprocessing import TweetPreprocessor, light_clean, normalize_tweet

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 4)

config.ensure_dirs()
config.set_seed()
FIG, MET = config.FIGURES_DIR, config.METRICS_DIR


def show(path, width=760):
    # Every plotting helper in src/ saves a PNG; display the saved file inline.
    display(Image(filename=str(path), width=width))


print(f"Project root: {ROOT}\nRandom seed: {config.SEED}")

## 1. Data Collection

### 1.1 Load and clean


In [ ]:
raw = load_raw()
df, cleaning_report = clean_dataset(raw)
print(f"Raw rows: {len(raw):,}  →  clean rows: {len(df):,}")
pd.Series(cleaning_report, name="count").to_frame()

In [ ]:
df[["tweet_id", "airline", "airline_sentiment", "airline_sentiment_confidence", "negativereason", "text"]].sample(8, random_state=config.SEED)

### 1.2 Summary statistics


In [ ]:
tables = summary_statistics(df)
display(Markdown("**Dataset overview**"), tables["overview"])
display(Markdown("**Sentiment distribution**"), tables["sentiment"])
display(Markdown("**Sentiment by airline**"), tables["by_airline"])
display(Markdown("**Negative-tweet reasons**"), tables["negative_reasons"])

In [ ]:
for name in ["sentiment_distribution", "sentiment_by_airline", "negative_reasons", "reasons_by_airline",
             "length_by_sentiment", "daily_volume"]:
    getattr(eda, f"fig_{name}")(df, FIG / f"eda_{name}.png")
    show(FIG / f"eda_{name}.png")

### 1.3 Train / validation / test split


In [ ]:
splits = split_data(df)
pd.DataFrame({part: getattr(splits, part)["airline_sentiment"].value_counts(normalize=True).reindex(config.LABELS)
              for part in ("train", "val", "test")}).T.assign(
    n=[len(splits.train), len(splits.val), len(splits.test)]).round(3)

## 2. Data Preprocessing


In [ ]:
pp = TweetPreprocessor()
examples = [
    "@united I wasn't happy, flights were DELAYED!!! #BadService http://t.co/xyz",
    "@SouthwestAir thank you sooooo much for the upgrade 😍 :)",
    "@AmericanAir can you tell me if flight 1234 to DFW is on time?",
    "@JetBlue never flying with you again. Lost my bag AND no one cares 😡",
]
pd.DataFrame({
    "original": examples,
    "normalised": [normalize_tweet(t) for t in examples],
    "final tokens": [pp.process(t) for t in examples],
    "Transformer input (light_clean)": [light_clean(t) for t in examples],
})

In [ ]:
# Preprocess the whole corpus once (batched POS tagging) and index by tweet_id.
t0 = time.perf_counter()
all_tokens = pp.process_corpus(df["text"])
tokens_by_id = dict(zip(df["tweet_id"], all_tokens))
print(f"Preprocessed {len(all_tokens):,} tweets in {time.perf_counter() - t0:.0f}s")


def docs(part):
    # Space-joined preprocessed tokens for one split, in that split's row order.
    return [" ".join(tokens_by_id[i]) for i in getattr(splits, part)["tweet_id"]]


X_train, X_val, X_test = docs("train"), docs("val"), docs("test")
y_train, y_val, y_test = (getattr(splits, p)["label"].values for p in ("train", "val", "test"))

raw_vocab = len(set(df["text"].str.lower().str.split().explode()))
clean_vocab = len({t for toks in all_tokens for t in toks})
pd.DataFrame({"value": [df["text"].str.split().str.len().sum(), sum(map(len, all_tokens)), raw_vocab, clean_vocab,
                        sum(len(t) == 0 for t in all_tokens)]},
             index=["Raw whitespace tokens", "Tokens after preprocessing", "Raw vocabulary (lower-cased)",
                    "Vocabulary after preprocessing", "Tweets empty after preprocessing"])

### 2.1 Distinctive terms per sentiment


In [ ]:
terms = eda.log_odds_dirichlet(all_tokens, df["label"].values)
eda.fig_distinctive_terms(terms, FIG / "eda_distinctive_terms.png")
show(FIG / "eda_distinctive_terms.png")

### 2.2 Does each preprocessing step help? (ablation)


In [ ]:
ablation_path = MET / "preprocessing_ablation.csv"
if ablation_path.exists():
    ablation = pd.read_csv(ablation_path)
    display(ablation)
else:
    print("Run `python scripts/run_pipeline.py --stages ablation` to produce this table.")

## 3. Feature Extraction


In [ ]:
tfidf = word_tfidf(ngram_range=(1, 2)).fit(X_train)
X_train_tfidf = tfidf.transform(X_train)
print(f"TF-IDF matrix: {X_train_tfidf.shape[0]:,} tweets × {X_train_tfidf.shape[1]:,} features; "
      f"density {X_train_tfidf.nnz / np.prod(X_train_tfidf.shape):.4%}")

idf = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out())
pd.DataFrame({"lowest IDF (most common)": idf.nsmallest(12).index,
              "highest IDF (rarest kept)": idf.nlargest(12).index})

In [ ]:
# Word2Vec: nearest neighbours show that the embedding has learned domain semantics.
w2v = Word2VecVectorizer().fit(X_train)
probe = ["delay", "rude", "great", "bag", "refund"]
pd.DataFrame({w: [f"{n} ({s:.2f})" for n, s in w2v.model_.wv.most_similar(w, topn=6)] for w in probe if w in w2v.model_.wv})

### 3.1 Visualising the feature space


In [ ]:
proj = project_feature_space(pd.Series(TweetPreprocessor.join(all_tokens)), df["label"])
eda.fig_feature_space(proj, FIG / "feature_space.png")
show(FIG / "feature_space.png")

## 4. Model Development

### 4.1 Training strategy for the classical models


In [ ]:
specs = get_model_specs()
pd.DataFrame([{"slug": s.slug, "model": s.name, "features": s.features,
               "search grid": {k: v for k, v in s.param_grid.items()}} for s in specs])

In [ ]:
MODEL_INFO_DIR = MET / "models"
X_fit, y_fit = X_train + X_val, np.concatenate([y_train, y_val])

fitted, infos, preds = {}, {}, {}
for spec in specs:
    info_path = MODEL_INFO_DIR / f"{spec.slug}.json"
    t0 = time.perf_counter()
    if RUN_GRID_SEARCH or not info_path.exists():
        model, info = tune_and_fit(spec, X_fit, y_fit)
        mode = "grid search"
    else:
        # Refit with the hyper-parameters the grid search selected (JSON stores tuples as lists).
        info = json.loads(info_path.read_text())
        params = {k: tuple(v) if isinstance(v, list) else v for k, v in info["best_params"].items()}
        model = spec.build().set_params(**params).fit(X_fit, y_fit)
        mode = "refit with tuned params"
    y_pred = model.predict(X_test)
    preds[spec.slug] = pd.DataFrame({"tweet_id": splits.test["tweet_id"], "y_true": y_test, "y_pred": y_pred})
    preds[spec.slug][ev.SCORE_COLUMNS] = decision_scores(model, X_test)
    fitted[spec.slug], infos[spec.slug] = model, info
    print(f"{spec.name:<32s} {mode:<24s} {time.perf_counter() - t0:5.1f}s  params={info['best_params']}")

pd.DataFrame(infos).T[["name", "features", "best_params", "cv_macro_f1_mean", "cv_macro_f1_std"]]

In [ ]:
# Verify held-out IDs/labels before comparing notebook and saved predictions.
saved = ev.load_predictions()
ev.validate_predictions(preds, splits.test)
ev.validate_predictions(saved, splits.test)
for slug, p in preds.items():
    if slug in saved:
        agree = (saved[slug].set_index("tweet_id")["y_pred"].reindex(p["tweet_id"]).values == p["y_pred"].values).mean()
        print(f"{slug:<14s} agreement with saved predictions: {agree:.2%}")
        if agree < 1:
            warnings.warn(f"{slug}: refit differs from saved predictions; notebook results reflect this refit. Rebuild the paper after running the notebook.")


### 4.2 Recurrent neural network: BiLSTM


In [ ]:
from src.rnn_model import RNNConfig
display(pd.Series(vars(RNNConfig()), name="BiLSTM hyper-parameters").astype(str).to_frame())

if TRAIN_BILSTM:
    import torch
    from src.rnn_model import predict_proba as rnn_proba, train_bilstm
    seq_pp = TweetPreprocessor(remove_stopwords=False)
    seq = {p: seq_pp.process_corpus(getattr(splits, p)["text"]) for p in ("train", "val", "test")}
    rnn, vocab, rnn_info = train_bilstm(seq["train"], y_train, seq["val"], y_val, RNNConfig())
    probs = rnn_proba(rnn, vocab, seq["test"], RNNConfig().max_len)
    rnn_info.update({"slug": "bilstm", "name": "BiLSTM (RNN)", "family": "Deep learning (RNN)"})
    MODEL_INFO_DIR.mkdir(parents=True, exist_ok=True)
    (MODEL_INFO_DIR / "bilstm.json").write_text(json.dumps(rnn_info, indent=2), encoding="utf-8")
    ev.save_predictions("bilstm", splits.test["tweet_id"], y_test, probs.argmax(1), probs)
    ev.plot_training_history(rnn_info["history"], "BiLSTM training curves", FIG / "training_bilstm.png")

### 4.3 Transformer: fine-tuned DistilRoBERTa


In [ ]:
from src.transformer_model import TransformerConfig
display(pd.Series(vars(TransformerConfig()), name="DistilRoBERTa hyper-parameters").astype(str).to_frame())

if TRAIN_TRANSFORMER:
    from src.transformer_model import predict_proba as tr_proba, train_transformer
    cfg = TransformerConfig()
    clean = {p: [light_clean(t) for t in getattr(splits, p)["text"]] for p in ("train", "val", "test")}
    tr_model, tokenizer, tr_info = train_transformer(clean["train"], y_train, clean["val"], y_val, cfg,
                                                     save_dir=config.MODELS_DIR / "distilroberta")
    probs = tr_proba(tr_model, tokenizer, clean["test"], cfg.max_length)
    tr_info.update({"slug": "distilroberta", "name": "DistilRoBERTa", "family": "Deep learning (Transformer)"})
    MODEL_INFO_DIR.mkdir(parents=True, exist_ok=True)
    (MODEL_INFO_DIR / "distilroberta.json").write_text(json.dumps(tr_info, indent=2), encoding="utf-8")
    ev.save_predictions("distilroberta", splits.test["tweet_id"], y_test, probs.argmax(1), probs)
    ev.plot_training_history(tr_info["history"], "DistilRoBERTa fine-tuning curves", FIG / "training_distilroberta.png")

In [ ]:
# Add the neural models' saved test predictions (same test tweets, same CSV schema).
for slug in ("bilstm", "distilroberta"):
    pred_path = config.PREDICTIONS_DIR / f"{slug}_test.csv"
    info_path = MODEL_INFO_DIR / f"{slug}.json"
    if pred_path.exists():
        preds[slug] = pd.read_csv(pred_path)
        infos[slug] = json.loads(info_path.read_text()) if info_path.exists() else {"name": slug}
        fig_path = FIG / f"training_{slug}.png"
        if fig_path.exists():
            show(fig_path, width=640)
    else:
        print(f"{slug}: no saved predictions yet. Run `python scripts/run_pipeline.py --stages "
              f"{'rnn' if slug == 'bilstm' else 'transformer'}` or set the TRAIN flag above.")

names = {slug: info.get("name", slug) for slug, info in infos.items()}
order = [m for m in MODEL_ORDER if m in preds]
print("Models in the comparison:", [names[m] for m in order])

## 5. Model Evaluation


In [ ]:
ev.validate_predictions(preds, splits.test)
table = ev.comparison_table(preds, names)
for col in ("family", "algorithm", "features", "cv_macro_f1_mean", "best_val_macro_f1", "inference_ms_per_1k"):
    table[col] = table["slug"].map(lambda s, c=col: infos.get(s, {}).get(c))
table.to_csv(MET / "model_comparison.csv", index=False)
best = table.iloc[0]["slug"]
cols = ["model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_macro_ci_low", "f1_macro_ci_high",
        "f1_weighted", "roc_auc_macro_ovr"]
display(Markdown(f"**Best model by macro-F1: {names[best]}**"))
table[cols].style.format(precision=3).highlight_max(subset=cols[1:], color="#d6e6fa")

In [ ]:
ev.plot_model_comparison(table, FIG / "model_comparison.png")
show(FIG / "model_comparison.png")

### 5.1 Per-class performance


In [ ]:
per_class = pd.concat({names[s]: ev.per_class_report(preds[s]["y_true"], preds[s]["y_pred"]) for s in order},
                      names=["model", "class"])
per_class.to_csv(MET / "per_class_metrics.csv")
display(per_class.round(3))
ev.plot_per_class_f1(preds, names, FIG / "per_class_f1.png", order)
show(FIG / "per_class_f1.png")

### 5.2 Confusion matrices (row-normalised: each row shows how that true class was predicted)


In [ ]:
ev.plot_confusion_grid(preds, names, FIG / "confusion_matrices.png", order)
show(FIG / "confusion_matrices.png", width=900)

### 5.3 ROC curves


In [ ]:
ev.plot_roc_per_class(preds[best], f"ROC curves – {names[best]}", FIG / "roc_best_model.png")
ev.plot_roc_by_model(preds, names, FIG / "roc_all_models.png", order)
show(FIG / "roc_best_model.png", width=560)
show(FIG / "roc_all_models.png", width=560)

### 5.4 Are the differences statistically significant? (McNemar's test vs. the best model)


In [ ]:
sig = ev.significance_table(preds, best, names)
sig.to_csv(MET / "mcnemar_tests.csv", index=False)
sig

In [ ]:
# Plain-language summary generated from the numbers above.
b = table.iloc[0]
lines = [f"* **{b['model']}** achieves macro-F1 **{b['f1_macro']:.3f}** "
         f"(95 % CI {b['f1_macro_ci_low']:.3f}–{b['f1_macro_ci_high']:.3f}) and accuracy **{b['accuracy']:.3f}**."]
classical = table[table["slug"].isin([s.slug for s in specs])]
if not classical.empty:
    c = classical.iloc[0]
    lines.append(f"* Best classical model: **{c['model']}**, macro-F1 {c['f1_macro']:.3f}.")
not_sig = sig.loc[~sig["significant_holm_at_0.05"], "comparison"].str.split(" vs ").str[1].tolist()
lines.append("* No detected paired error-rate difference after Holm adjustment (p ≥ 0.05): "
             + (", ".join(not_sig) if not_sig else "none; adjusted tests detect differences for every comparison") + ".")
weakest = per_class.xs(names[best], level="model")["f1"].idxmin()
lines.append(f"* Hardest class for the best model: **{weakest}** "
             f"(F1 {per_class.loc[(names[best], weakest), 'f1']:.3f}).")
display(Markdown("\n".join(lines)))

## Observations

These observations describe the verified saved experiment. If the notebook is rerun, use the resulting evaluation tables for current scores.

* **Dataset composition:** cleaning leaves 14,485 unique tweet IDs. Sentiments are 62.7% negative, 21.2% neutral and 16.1% positive. This imbalance makes macro-F1 more informative than accuracy alone.
* **Model comparison:** word+character TF-IDF SVM leads the six evaluated configurations with 81.41% accuracy, macro precision 0.7688, macro recall 0.7632 and macro-F1 0.7658. BiLSTM follows with macro-F1 0.7616; averaged Word2Vec with Logistic Regression achieves 0.6704.
* **Class-level performance:** the leading SVM achieves F1-scores of 0.8781 for negative, 0.6607 for neutral and 0.7585 for positive tweets. Neutral is its most difficult class, although BiLSTM has slightly higher neutral-class F1 (0.6654).
* **Preprocessing:** the ablation shows lower CV macro-F1 after stop-word removal and lemmatization than after normalization and tokenization alone. Removing negations lowers the score further. Preprocessing choices should therefore be evaluated rather than assumed beneficial.
* **Complaint themes:** customer-service issues account for 31.7% of recorded negative reasons and late flights for 18.2%. These are existing annotation summaries, not reasons predicted by the sentiment classifier or proof of operational causes.
* **Statistical evidence:** Holm-adjusted McNemar tests detect an error-rate difference between the leading SVM and BiLSTM (p = 0.0466), but not the word-only SVM (p = 0.0583). These exploratory tests concern classification errors, not macro-F1 superiority or model equivalence.


## 6. Interpretation and Business Insights


In [ ]:
coefs = insights.top_coefficients(fitted["lr_tfidf"])
insights.fig_top_coefficients(coefs, "What drives the Logistic Regression model (top TF-IDF coefficients)",
                              FIG / "insight_top_coefficients.png")
show(FIG / "insight_top_coefficients.png")

In [ ]:
best_merged = splits.test.merge(preds[best], on="tweet_id", validate="one_to_one")
kpis = insights.airline_kpis(best_merged)
kpis.to_csv(MET / "airline_kpis.csv")
display(kpis)
insights.fig_airline_kpis(kpis, names[best], FIG / "insight_airline_nss.png")
show(FIG / "insight_airline_nss.png")

swapped = kpis.index[kpis["actual_rank"] != kpis["predicted_rank"]].tolist()
display(Markdown(
    f"Predicted vs. human-labelled Net Sentiment Score: mean absolute error **{kpis['abs_error_nss'].mean():.1f} points** "
    f"(max {kpis['abs_error_nss'].max():.1f}). Airline ranking reproduced "
    + ("exactly." if not swapped else f"except for **{' / '.join(swapped)}**, whose actual scores differ by only "
       f"{kpis.loc[swapped, 'actual_nss'].max() - kpis.loc[swapped, 'actual_nss'].min():.1f} points.")))

In [ ]:
triage, curve = insights.negative_triage(best_merged)
triage.to_csv(MET / "negative_triage_operating_points.csv", index=False)
display(triage)
insights.fig_negative_pr(curve, triage, names[best], FIG / "insight_negative_pr.png")
show(FIG / "insight_negative_pr.png", width=600)

In [ ]:
display(Markdown("**Recall on negative tweets, by complaint reason**"))
display(insights.recall_by_reason(best_merged))

compare = list(dict.fromkeys(s for s in (best, "svm_tfidf", "nb_tfidf") if s in preds))
acc_conf = insights.accuracy_by_confidence({s: splits.test.merge(preds[s], on="tweet_id") for s in compare}, names)
acc_conf.to_csv(MET / "accuracy_by_confidence.csv", index=False)
display(Markdown("**Accuracy by annotator confidence**"), acc_conf)
insights.fig_accuracy_by_confidence(acc_conf, FIG / "insight_accuracy_by_confidence.png")
show(FIG / "insight_accuracy_by_confidence.png", width=640)

In [ ]:
display(Markdown("**The best model's most confident mistakes**"))
insights.confident_errors(best_merged, n=10)

### 6.1 Final Insights and Recommendations

The results support evaluating word+character TF-IDF SVM as a practical baseline for airline feedback classification. Its leading overall macro-F1 and accuracy make it a useful candidate, while BiLSTM's stronger neutral recall illustrates that model choice also depends on the business cost of different errors.

1. **Pilot complaint triage with human review.** At an exploratory test-selected operating point, the SVM catches approximately 95% of negative tweets with 80.4% precision and flags 74.1% of test tweets. Select an operational threshold on validation or out-of-fold scores, then evaluate it on fresh data before using these figures for staffing.
2. **Investigate frequent complaint categories.** Use the annotated customer-service and delay themes to guide operational investigation. Measure whether any intervention improves outcomes; this analysis does not establish causal effects or return on investment.
3. **Improve neutral-class handling.** Review ambiguous and mixed-sentiment examples, assess annotation consistency and compare alternatives using class-specific metrics. Keep human oversight for consequential decisions. SVM margins require calibration before being interpreted as probabilities.
4. **Strengthen validation and reproducibility.** Use grouped text/user or chronological splits, record full search grids and library versions, and set the interpreter hash seed before startup. Compare pipelines with equal training-data budgets when isolating algorithm performance.
5. **Monitor performance over time.** The February 2015 Twitter sample does not represent all airline customers or current language. Audit new predictions and monitor drift. Transformer evaluation and aspect-based sentiment analysis remain potential extensions whose benefits must be measured.

### 6.2 Limitations

Thirteen distinct lowercased texts occur in both development and test sets despite disjoint tweet IDs. Classical models fit on 85% of the corpus, while neural models train on 70% and select checkpoints on 15%. Two saved search candidate counts differ from the current code grids, and their historical grid definitions are unavailable. These constraints limit independence, controlled model comparisons and exact reconstruction of the original searches. Airline summaries also inherit the existing Delta-to-JetBlue relabeling assumption.

## 7. Conclusion

This study compares six sentiment-classification configurations across Naive Bayes, Logistic Regression, Linear SVM and BiLSTM using the existing airline dataset. Word+character TF-IDF SVM performs best in the saved overall comparison, achieving 81.41% accuracy and macro-F1 0.7658. Neutral tweets remain the leading model's weakest class, and more complex models do not consistently improve overall performance in this experiment.

Sentiment predictions could support complaint routing and feedback monitoring, while existing complaint annotations identify themes for investigation. The findings justify a carefully evaluated pilot rather than immediate production claims. Independent threshold evaluation, stronger data splitting and ongoing human audits are the main next steps.


## References
* Dietterich, T. G. (1998). Approximate statistical tests for comparing supervised classification learning algorithms. *Neural Computation, 10*(7), 1895–1923.
* Joachims, T. (1998). Text categorization with support vector machines: Learning with many relevant features. In *ECML-98* (pp. 137–142). Springer.
* Liu, Y., et al. (2019). RoBERTa: A robustly optimized BERT pretraining approach. *arXiv:1907.11692*.
* McCallum, A., & Nigam, K. (1998). A comparison of event models for naive Bayes text classification. *AAAI-98 Workshop on Learning for Text Categorization*.
* Mikolov, T., Chen, K., Corrado, G., & Dean, J. (2013). Efficient estimation of word representations in vector space. *arXiv:1301.3781*.
* Monroe, B. L., Colaresi, M. P., & Quinn, K. M. (2008). Fightin' words: Lexical feature selection and evaluation for identifying the content of political conflict. *Political Analysis, 16*(4), 372–403.
* Salton, G., & Buckley, C. (1988). Term-weighting approaches in automatic text retrieval. *Information Processing & Management, 24*(5), 513–523.
* Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). DistilBERT, a distilled version of BERT. *arXiv:1910.01108*.
* Schuster, M., & Paliwal, K. K. (1997). Bidirectional recurrent neural networks. *IEEE Transactions on Signal Processing, 45*(11), 2673–2681.
* Figure Eight. (2015). *Twitter US Airline Sentiment* [Data set]. Kaggle. https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment